# Gold: vistas KPI

Ejecuta `sql/gold/kpi_views.sql`: 6 vistas que centralizan reglas de negocio (ej. "win rate solo sobre oportunidades cerradas") en un solo lugar, para que Power BI/Superset/lo que sea lean el numero ya correcto en vez de recalcularlo cada herramienta por su cuenta.

**Requiere que `02_estrella_academica`, `03_estrella_billing` y `04_estrella_crm` ya hayan corrido.**

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from pathlib import Path
from utils.db import get_psycopg2_connection, get_engine

engine = get_engine()
SQL_GOLD = Path("/home/jovyan/work/sql/gold")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

In [2]:
run_sql_file(SQL_GOLD / "kpi_views.sql")

OK: kpi_views.sql ejecutado


## Verificar contra las cifras ya conocidas (`notebooks/analysis/01_insights.ipynb`)

In [3]:
print("win_rate_by_industry -- esperado: finance 68.9%, education 57.0%")
pd.read_sql("SELECT * FROM gold.vw_win_rate_by_industry ORDER BY win_rate_pct DESC", engine)

win_rate_by_industry -- esperado: finance 68.9%, education 57.0%


,industry,oportunidades_cerradas,ganadas,win_rate_pct
0,finance,90,62,68.9
1,energy,61,38,62.3
2,tech,144,89,61.8
3,services,72,44,61.1
4,manufacturing,82,50,61.0
5,retail,142,84,59.2
6,health,109,64,58.7
7,education,79,45,57.0


In [4]:
print("churn_by_segment -- esperado: retail 15.2%, smb 15.0%, enterprise 13.1%")
pd.read_sql("SELECT * FROM gold.vw_churn_by_segment ORDER BY churn_rate_pct DESC", engine)

churn_by_segment -- esperado: retail 15.2%, smb 15.0%, enterprise 13.1%


,segment,suscripciones,canceladas,churn_rate_pct
0,retail,10528,1595,15.2
1,smb,3301,494,15.0
2,enterprise,1171,153,13.1


In [5]:
print("academic_performance_by_department -- esperado: ~74.7-75.1 en todos")
pd.read_sql("SELECT * FROM gold.vw_academic_performance_by_department ORDER BY promedio DESC", engine)

academic_performance_by_department -- esperado: ~74.7-75.1 en todos


,department,inscripciones,promedio,pct_aprobados
0,biology,3411,75.1,95.9
1,cs,3662,75.1,95.7
2,chemistry,2505,75.0,95.3
3,math,1765,75.0,95.5
4,economics,2612,74.8,95.6
5,history,2378,74.8,95.5
6,literature,2547,74.8,94.9
7,physics,3906,74.7,95.0


In [6]:
print("lead_conversion_by_source -- esperado: cold_call 14.7%, web 8.5%")
pd.read_sql("SELECT * FROM gold.vw_lead_conversion_by_source ORDER BY conversion_rate_pct DESC", engine)

lead_conversion_by_source -- esperado: cold_call 14.7%, web 8.5%


,source,leads,convertidos,conversion_rate_pct
0,cold_call,204,30,14.7
1,referral,402,46,11.4
2,event,302,32,10.6
3,ads,282,28,9.9
4,web,810,69,8.5


In [7]:
print("dso_by_payment_method -- esperado: 22.3-22.9 dias, casi igual en los 4")
pd.read_sql("SELECT * FROM gold.vw_dso_by_payment_method ORDER BY dias_promedio_pago DESC", engine)

dso_by_payment_method -- esperado: 22.3-22.9 dias, casi igual en los 4


,method,pagos,dias_promedio_pago
0,cash,3969,22.9
1,bank_transfer,24054,22.6
2,card,43990,22.6
3,paypal,7987,22.3


In [8]:
print("retention_by_student_status -- esperado: estudiante 14.5% cancela, no-estudiante 15.4%")
pd.read_sql("SELECT * FROM gold.vw_retention_by_student_status", engine)

retention_by_student_status -- esperado: estudiante 14.5% cancela, no-estudiante 15.4%


,is_student,suscripciones,pct_activas,pct_canceladas
0,False,7427,74.9,15.4
1,True,7573,75.4,14.5
